# 从三点拟合到 PCA
先修：矩阵乘法、内积和正文的正交投影。按顺序运行，可直接修改数据；本实验不锁定输入。

我们依次研究拟合、分解、压缩与扰动。先预测：一条直线无法穿过三点时，残差能否与设计矩阵的每列垂直？

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)
X = np.array([[1., 0.], [1., 1.], [1., 2.]])
y = np.array([1., 2., 2.])
beta = np.linalg.lstsq(X, y, rcond=None)[0]
fitted = X @ beta
residual = y - fitted
print('系数、拟合、残差：', beta, fitted, residual, sep='\n')
print('残差内积：', X.T @ residual)

正文通过两条正规方程相减得到斜率 $1/2$、截距 $7/6$，残差为 $(-1/6,1/3,-1/6)$。下面把误差画出来。竖直距离平方和就是最小化的目标；观测空间中的正交性由上一格的内积体现。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(X[:, 1], y, label='observed')
ax.plot(X[:, 1], fitted, label='fitted')
ax.vlines(X[:, 1], y, fitted, color='tab:red', label='residual')
ax.set(xlabel='time', ylabel='response'); ax.legend(); plt.show()
print('与解析系数的差：', beta - np.array([7/6, 1/2]))

## 同一个解的不同坐标
QR 把投影坐标变成上三角方程。SVD 逐个方向除以奇异值。观察中间矩阵，而不仅是最终答案。

In [ ]:
Q, R = np.linalg.qr(X, mode='reduced')
beta_qr = np.linalg.solve(R, Q.T @ y)
U, singular, Vt = np.linalg.svd(X, full_matrices=False)
beta_svd = Vt.T @ ((U.T @ y) / singular)
rank_one = singular[0] * np.outer(U[:, 0], Vt[0])
print('Q, R：', Q, R, sep='\n')
print('QR / SVD 系数：', beta_qr, beta_svd)
print('奇异值：', singular)
print('秩一误差与第二奇异值：', np.linalg.norm(X-rank_one, 2), singular[1])
beta_normal = np.linalg.solve(X.T @ X, X.T @ y)
beta_pinv = np.linalg.pinv(X) @ y
print("正规方程 / 伪逆：", beta_normal, beta_pinv)

## PCA 与训练样本白化
下面生成总体协方差为 $\left(\begin{smallmatrix}2&1\\1&2\end{smallmatrix}\right)$ 的二维样本。总体首方向是 $(1,1)/\sqrt2$，解释比例为 75%；样本结果会有随机误差。改变样本数，看它怎样变化。

In [ ]:
rng = np.random.default_rng(2026)
population = np.array([[2., 1.], [1., 2.]])
train = rng.multivariate_normal([0, 0], population, size=80)
center = train.mean(axis=0)
Z = train - center
S = Z.T @ Z / (len(Z)-1)
eigenvalues, directions = np.linalg.eigh(S)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, directions = eigenvalues[order], directions[:, order]
scores = Z @ directions
reconstructed = np.outer(scores[:, 0], directions[:, 0])
white = scores / np.sqrt(eigenvalues)
new = rng.multivariate_normal([0, 0], population, size=2000)
white_new = ((new-center) @ directions) / np.sqrt(eigenvalues)
print('解释比例：', eigenvalues / eigenvalues.sum())
print('舍弃方向的样本平方误差：', np.sum((Z-reconstructed)**2)/(len(Z)-1))
print('训练样本白化协方差：', np.cov(white, rowvar=False))
print('新样本白化协方差：', np.cov(white_new, rowvar=False))
fig, ax = plt.subplots()
ax.scatter(Z[:, 0], Z[:, 1], alpha=.5, label='centered')
ax.scatter(reconstructed[:, 0], reconstructed[:, 1], s=12, label='rank one')
ax.axis('equal'); ax.legend(); plt.show()

训练协方差为单位阵是代数恒等式；新样本没有这一保证。这里训练与新样本来自同一总体，差别仍会由训练估计误差产生。新样本数增大只能减少新样本自身的抽样误差，不能消除已固定的训练变换误差。

In [ ]:
D = np.diag([1., 1e-6])
b = np.array([1., 0.])
db = np.array([0., 1e-6])
exact = np.linalg.solve(D, b)
changed = np.linalg.solve(D, b+db)
print('输入相对变化：', np.linalg.norm(db)/np.linalg.norm(b))
print('解相对变化：', np.linalg.norm(changed-exact)/np.linalg.norm(exact))
print('条件数：', np.linalg.cond(D))

## 自己试一试
1. 把设计矩阵的第二列改成第一列。系数和拟合值是否仍唯一？
2. 只保留较小奇异值，秩一近似误差怎样变化？
3. 将上格扰动换到第一坐标，是否仍放大一百万倍？

先作答，再运行下格参考。

In [ ]:
duplicate = np.column_stack([np.ones(3), np.ones(3)])
minimum = np.linalg.lstsq(duplicate, y, rcond=None)[0]
alternative = minimum + np.array([1., -1.])
small_only = singular[1] * np.outer(U[:, 1], Vt[1])
print('两个系数：', minimum, alternative)
print('拟合差：', duplicate @ (minimum-alternative))
print('只保留较小项的误差：', np.linalg.norm(X-small_only, 2))
db_first = np.array([1e-6, 0.])
print('第一方向放大：', np.linalg.norm(np.linalg.solve(D, db_first))/np.linalg.norm(db_first))

系数可沿零空间移动，拟合仍唯一。保留较小项时误差等于最大奇异值；第一方向放大为 1。高条件数描述最坏方向，不能解释为每次误差都同样放大。